# 00 - Baseline: Original MRNet (reference)

**Owner:** Ilaria  |  **Plane:** sagittal  |  **Tasks:** abnormal / acl / meniscus

This notebook reproduces the original MRNet baseline from the paper:
> Bien et al., *Deep-learning-assisted diagnosis for knee MRI*, PLOS Medicine 2018.

- **Architecture:** Pretrained AlexNet per slice, max-pooling across slices, FC head (binary output).
This is almost identical to the tutorial solution, but trained through our group's shared pipeline
(`data_pipeline.py`, `training_utils.py`) so that all subsequent models are compared under identical conditions.
- **Plane:** Sagittal only (consistent with the main project sweep).
- **Tasks:** All three (`abnormal`, `acl`, `meniscus`).

> Import from `src/` only - no model logic should live in the notebook.

> **GPU equivalent:** [`for-gpu/run_baseline.py`](../for-gpu/run_baseline.py) runs
> this exact pipeline headless on the DGX/A6000 boxes. Both build the model with
> `src.model_factory.build_baseline_model()` and train through
> `src.training_utils.run_training`. Trained weights -> `model_checkpoints/`;
> metrics/figures -> `for-gpu/results/`.

> **Data flow:** `build_dataloaders()` -> `MRNetDataset` -> `DataLoader` -> `run_training()`. See [`../GRADING.md`](../GRADING.md) for the rubric quick-map.

## 0. Setup

In [2]:
# ============================================================
# Colab bootstrap - run this cell first.
# ============================================================
import sys
from pathlib import Path

# Install dependencies not available by default in Colab
!pip install tensorboardX --quiet

# Mount Google Drive when running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# Locate the "codes" folder (the one containing src/ and notebooks/).
if IN_COLAB:
    # Shared Drive path, consistent for all teammates (no manual shortcuts needed).
    CODES_DIR = Path('/content/drive/Shareddrives/MRNet_Group3/codes')
else:
    # Local run: this notebook lives in codes/notebooks/, so go up one level.
    CODES_DIR = Path.cwd().parent

assert (CODES_DIR / 'src').exists(), (
    f"Could not find src/ under {CODES_DIR}. "
    "Edit CODES_DIR above to point to your 'codes' folder.")

# Make `import src...` work.
if str(CODES_DIR) not in sys.path:
    sys.path.insert(0, str(CODES_DIR))

# Shared config resolves the data folder (sibling of "codes").
from src import config
print('CODES_DIR:', config.CODES_DIR)
print('DATA_DIR :', config.DATA_DIR)
assert config.DATA_DIR.exists(), (
    f"Data folder not found at {config.DATA_DIR}. Put the MRNet 'data' folder "
    "next to 'codes', or set os.environ['MRNET_DATA_DIR'] before importing config."
)
print('Setup OK. Data folder found.')


Mounted at /content/drive
CODES_DIR: /content/drive/Shareddrives/MRNet_Group3/codes
DATA_DIR : /content/drive/Shareddrives/MRNet_Group3/data
Setup OK. Data folder found.


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.models import AlexNet_Weights

# Our shared pipeline modules (imported via src, per the bootstrap cell above)
from src.data_pipeline import build_dataloaders, set_seed
from src.training_utils import run_training, load_checkpoint

print('All imports OK.')
print('CUDA available:', torch.cuda.is_available())


All imports OK.
CUDA available: False


In [4]:
# Shared configuration
# Paths and task/plane lists come from config.py so all teammates reproduce
# the same splits; only training hyperparameters are set locally here.

CHECKPOINT_DIR = config.CHECKPOINTS_DIR / 'baseline'
PLANE = config.DEFAULT_PLANE # sagittal only for the main sweep
TASKS = config.TASKS # ('abnormal', 'acl', 'meniscus')

# Training hyperparameters (baseline values from the tutorial)
LR = 1e-5
WEIGHT_DECAY = 0.1
NUM_EPOCHS = 50
ACCUM_STEPS = 8  # effective batch size = 8 (each exam is 1 sample)
EARLY_PATIENCE = 10

# Augmentation preset for the training set (see data_pipeline.AUGMENTATION_PRESETS).
# "medium" (rotation=20, translate=0.08) is the closest built-in match to the
# tutorial's own values (25 degrees, 0.11); validation always uses "none" internally.
TRAIN_AUGMENT = 'medium'

set_seed(config.SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


Using device: cpu


## 1. Baseline Model: AlexNet + Max-Pool

The `AlexNetBaseline` class replicates the architecture from Bien et al. (2018):

1. A pretrained AlexNet (ImageNet weights) processes each MRI slice independently,
   producing a 256-dimensional feature vector per slice.
2. Max-pooling across all slices collapses the variable-length slice stack into a
   single exam-level representation. This is the simplest aggregation strategy and
   serves as our baseline, attention-based pooling (used in later models) is expected
   to improve on this.
3. A single fully-connected layer maps the 256-d vector to a scalar logit for binary
   classification (sigmoid applied at inference time).

In [5]:
# The baseline architecture now lives in src.model_factory.AlexNetBaseline, so
# this notebook, run_baseline.py and eval_external.py all build the IDENTICAL
# model. build_baseline_model() = pretrained AlexNet per slice -> max-pool across
# slices -> single linear logit (Bien et al. 2018). See that class for details.
from src.model_factory import build_baseline_model

In [6]:
# Quick sanity check: verify the model produces the correct output shape
# before committing to a full training run.
dummy_exam = torch.zeros(1, 30, 3, 256, 256) # 1 exam, 30 slices
test_model = build_baseline_model()
with torch.no_grad():
    out = test_model(dummy_exam)
print('Output shape:', out.shape) # expected: torch.Size([1, 1])
assert out.shape == (1, 1), 'Unexpected output shape!'
print('Model architecture check passed.')


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:03<00:00, 62.0MB/s]


Output shape: torch.Size([1, 1])
Model architecture check passed.


## 2. Data Loading

We use the shared `data_pipeline.py` to ensure identical
preprocessing, augmentation, and train/val/test splits across all notebooks.

`build_dataloaders()` builds the dataset and DataLoader for one task/plane in a
single call, using a named augmentation preset (`TRAIN_AUGMENT` above) for the
training split. The validation split always uses `"none"` internally, so we don't
need to construct or pass our own transform objects (the original
`torchvision.transforms` pipeline doesn't apply here, since `MRNetDataset` works
on the raw normalized `.npy` arrays rather than PIL images).

Augmentations under the `"medium"` preset (consistent in spirit with the tutorial
baseline, see `AUGMENTATION_PRESETS` in `data_pipeline.py` for exact values):
- Random rotation
- Random horizontal flip
- Random affine translation and scale
- Intensity / contrast adjustment

No augmentation is applied to the validation set.

In [7]:
# Build one DataLoader pair per task using our shared data pipeline.
# build_dataloaders() handles the train/val split, label loading, augmentation,
# and normalization; root_dir defaults to config.DATA_DIR when omitted.
task_loaders = {}
for task in TASKS:
    train_loader, val_loader = build_dataloaders(
        task=task,
        plane=PLANE,
        train_augment=TRAIN_AUGMENT,
        batch_size=1, # one exam at a time (variable slice count)
        num_workers=0)
    task_loaders[task] = (train_loader, val_loader)
    print(f'{task}: {len(train_loader)} train exams, {len(val_loader)} val exams')


abnormal: 1130 train exams, 120 val exams
acl: 1130 train exams, 120 val exams
meniscus: 1130 train exams, 120 val exams


## 3. Training

We train a separate model for each of the three tasks using `run_training()` from
`training_utils.py`. Each task produces an independent binary classifier.

**Optimiser:** Adam with `lr=1e-5`, `weight_decay=0.1` same as the tutorial baseline.

**LR scheduler:** `ReduceLROnPlateau` on validation loss (patience=4, factor=0.3).

**Loss:** Class-weighted BCE, computed automatically from training label distribution
inside `run_training()` (improvement over the tutorial's per-sample weighting).

In [8]:
# ============================================================
# Resumable training wrapper.
# Reuses train_one_epoch / validate_one_epoch / compute_pos_weight
# directly, and adds: (1) a "last state" checkpoint after every epoch,
# (2) automatic resume from that checkpoint if present.
# ============================================================
import os
from src.training_utils import (
    compute_pos_weight, train_one_epoch, validate_one_epoch)

def run_training_resumable(model, train_loader, val_loader, optimizer, scheduler,
                            device, num_epochs=50, accumulation_steps=8,
                            early_stopping_patience=10, checkpoint_dir="checkpoints",
                            task_name="task"):
    os.makedirs(checkpoint_dir, exist_ok=True)
    best_ckpt_path = os.path.join(checkpoint_dir, f"best_{task_name}.pth")
    last_ckpt_path = os.path.join(checkpoint_dir, f"last_{task_name}.pth")

    history = {
        "train_loss": [], "train_auc": [],
        "val_loss":   [], "val_auc":   [],
        "val_metrics": []}

    start_epoch = 0
    best_val_auc = 0.0
    epochs_no_improve = 0
    pos_weight = None

    # Resume from "last state" checkpoint, if present
    if os.path.exists(last_ckpt_path):
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        best_val_auc = ckpt["best_val_auc"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        history = ckpt["history"]
        pos_weight = ckpt["pos_weight"].to(device)
        print(f"[{task_name}] Resuming from epoch {start_epoch} "
              f"(best val AUC so far: {best_val_auc:.4f})")

    if pos_weight is None:
        print(f"\n[{task_name}] Computing class weights from training data...")
        pos_weight = compute_pos_weight(train_loader, device)

    for epoch in range(start_epoch, num_epochs):

        train_loss, train_metrics = train_one_epoch(
            model, train_loader, optimizer, pos_weight, device, accumulation_steps)
        val_loss, val_metrics = validate_one_epoch(
            model, val_loader, pos_weight, device)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_auc"].append(train_metrics["auc"])
        history["val_loss"].append(val_loss)
        history["val_auc"].append(val_metrics["auc"])
        history["val_metrics"].append(val_metrics)

        print(
            f"[{task_name}] epoch: {epoch:3d} | "
            f"train loss: {train_loss:.4f} | train auc: {train_metrics['auc']:.4f} | "
            f"val loss: {val_loss:.4f} | val auc: {val_metrics['auc']:.4f} | "
            f"val f1: {val_metrics['f1']:.4f} | "
            f"sens: {val_metrics['sensitivity']:.4f} | spec: {val_metrics['specificity']:.4f}")
        print("-" * 80)

        improved = val_metrics["auc"] > best_val_auc
        if improved:
            best_val_auc = val_metrics["auc"]
            epochs_no_improve = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_auc": best_val_auc,
                "val_metrics": val_metrics}, best_ckpt_path)
            print(f"Checkpoint saved: {best_ckpt_path} (val AUC: {best_val_auc:.4f})")
        else:
            epochs_no_improve += 1

        # "Last state" checkpoint: written every epoch, enables resume
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_auc": best_val_auc,
            "epochs_no_improve": epochs_no_improve,
            "history": history,
            "pos_weight": pos_weight.cpu()}, last_ckpt_path)

        if epochs_no_improve >= early_stopping_patience:
            print(
                f"\n[{task_name}] Early stopping triggered after {epoch + 1} epochs "
                f"({early_stopping_patience} epochs without AUC improvement).")
            break

    print(f"\n[{task_name}] Training complete. Best val AUC: {best_val_auc:.4f}")

    # Clean up "last state" file once a task finishes successfully
    if os.path.exists(last_ckpt_path):
        os.remove(last_ckpt_path)

    return history

In [ ]:
# Train the baseline model on all three tasks.
# Results (best checkpoint per task) are saved to CHECKPOINT_DIR.

baseline_histories = {}

# Only ACL - change for final run
for task in TASKS:
    if task != "acl": # remove this if statement to run all tasks
        print(f"Skipping {task}.")
        continue
    else:
        print(f'\n{"=" * 70}')
        print(f'Training baseline - task: {task.upper()}')
        print(f'{"=" * 70}')

        train_loader, val_loader = task_loaders[task]

        # Fresh model for each task
        model = build_baseline_model().to(DEVICE)

        # Optimiser: Adam with tutorial baseline hyperparameters
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        # LR scheduler: reduce on val loss plateau, same as tutorial
        scheduler = ReduceLROnPlateau(
            optimizer, patience=4, factor=0.3, threshold=1e-4)

        # run_training() handles the full loop: class-weighted BCE, gradient
        # accumulation, early stopping, and checkpoint saving.
        #history = run_training( # uses base function from training_utils.py
        history = run_training_resumable( # saves and reloads checkpoints
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=DEVICE,
            num_epochs=NUM_EPOCHS,
            accumulation_steps=ACCUM_STEPS,
            early_stopping_patience=EARLY_PATIENCE,
            checkpoint_dir=str(CHECKPOINT_DIR),
            task_name=f'baseline_{task}')

        baseline_histories[task] = history


Skipping abnormal.

Training baseline - task: ACL

[baseline_acl] Computing class weights from training data...
  Class weight | neg: 922, pos: 208, pos_weight: 4.4327
[baseline_acl] epoch:   0 | train loss: 1.1066 | train auc: 0.6378 | val loss: 1.5614 | val auc: 0.8131 | val f1: 0.7083 | sens: 0.6296 | spec: 0.8788
--------------------------------------------------------------------------------
Checkpoint saved: /content/drive/Shareddrives/MRNet_Group3/codes/results/checkpoints/baseline/best_baseline_acl.pth (val AUC: 0.8131)
[baseline_acl] epoch:   1 | train loss: 0.9528 | train auc: 0.7617 | val loss: 1.4649 | val auc: 0.8367 | val f1: 0.7071 | sens: 0.6481 | spec: 0.8485
--------------------------------------------------------------------------------
Checkpoint saved: /content/drive/Shareddrives/MRNet_Group3/codes/results/checkpoints/baseline/best_baseline_acl.pth (val AUC: 0.8367)
[baseline_acl] epoch:   2 | train loss: 0.9139 | train auc: 0.7891 | val loss: 1.4535 | val auc: 0.8

## 4. Results Summary

Collect the best validation AUC per task and print a summary table.
These numbers are the reference baseline for the entire project,
every subsequent model should be compared against them.

In [ ]:
import pandas as pd

rows = []
for task in TASKS:
    history = baseline_histories[task]
    best_epoch = int(np.argmax(history['val_auc']))
    best_metrics = history['val_metrics'][best_epoch]
    rows.append({
        'Task': task,
        'Best Epoch': best_epoch,
        'Val AUC': best_metrics['auc'],
        'Val F1': best_metrics['f1'],
        'Sensitivity': best_metrics['sensitivity'],
        'Specificity': best_metrics['specificity'],
        'Accuracy': best_metrics['accuracy']})

results_df = pd.DataFrame(rows).set_index('Task')
print('\nBASELINE RESULTS (AlexNet + Max-Pool, Sagittal)')
print(results_df.to_string())


In [ ]:
import matplotlib.pyplot as plt

# Plot training curves (val AUC over epochs) for all three tasks.
# Useful for the report to show convergence behaviour of the baseline.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, task in zip(axes, TASKS):
    history = baseline_histories[task]
    epochs = range(len(history['val_auc']))

    ax.plot(epochs, history['train_auc'], label='Train AUC', linewidth=2)
    ax.plot(epochs, history['val_auc'], label='Val AUC', linewidth=2)
    ax.set_title(f'Baseline: {task.capitalize()}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AUC')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('AlexNet Baseline: Training Curves (Sagittal)', fontsize=13)
plt.tight_layout()

# Save figure for the report
fig_path = config.GPU_RESULTS_DIR / 'baseline_training_curves.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figure saved to: {fig_path}')
plt.show()


## 5. Save Results to CSV

Export the baseline results table to a CSV for easy access.

In [ ]:
csv_path = config.GPU_RESULTS_DIR / 'baseline_results.csv'
results_df.to_csv(csv_path)
print(f'Results saved to: {csv_path}')


## Notes for the Report

- **Model section**: explain that `__init__` defines the architecture (AlexNet backbone,
  FC head) and `forward` defines the computation graph (per-slice features, max-pool, classifier).
  Note that the original 1000-class ImageNet head was replaced with a single output node.
- **Why max-pool?** It is a simple, parameter-free aggregation: the model takes the most
  activated slice representation. It cannot learn which slices are more informative,
  this is the limitation that attention-based pooling addresses in later models.
- **Reference:** Bien, N. et al. (2018). Deep-learning-assisted diagnosis for knee
  magnetic resonance imaging. PLOS Medicine, 15(11).